In [1]:
!nvidia-smi

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   -- ------------------------------------- 0.8/11.6 MB 4.8 MB/s eta 0:00:03
   -------- ------------------------------- 2.4/11.6 MB 6.4 MB/s eta 0:00:02
   ------------ --------------------------- 3.7/11.6 MB 6.4 MB/s eta 0:00:02
   ----------------- ---------------------- 5.0/11.6 MB 6.3 MB/s eta 0:00:02
   --------------------- ------------------ 6.3/11.6 MB 6.3 MB/s eta 0:00:01
   --------------------------- ------------ 7.9/11.6 MB 6.4 MB/s eta 0:00:01
   ------------------------------- -------- 9.2/11.6 MB 6.5 MB/s eta 0:00:01
   ------------------------------------ --- 10.7/11.6 MB 6.6 MB/s eta 


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip install evaluate

In [ ]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import evaluate
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import torch
nltk.download("punkt")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
model_cktpt = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_cktpt)
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_cktpt).to(device)

In [ ]:
ds = load_dataset("knkarthick/samsum")
print(ds)

In [ ]:
split_lengths = [len(ds[split]) for split in ds]
print(f"Split lengths: {split_lengths}")
print(f"Features: {ds['train'].column_names}")
print("\nDialogue:")
print(ds['test'][1]['dialogue'])
print("\nSummary:")
print(ds['test'][1]['summary'])


In [ ]:
def convert_examples_to_features(example_batch):

    input_encoding = tokenizer(
        example_batch['dialogue'],
        max_length=1024,
        truncation=True
    )

    target_encoding = tokenizer(
        text_target=example_batch['summary'],
        max_length=1024,
        truncation=True
    )

    return {
        'input_ids': input_encoding['input_ids'],
        'attention_mask': input_encoding['attention_mask'],
        'labels': target_encoding['input_ids']
    }

In [ ]:
dataset_samsum_pt = ds.map(convert_examples_to_features, batched=True)

In [ ]:
print(dataset_samsum_pt)

In [ ]:
print(dataset_samsum_pt['train'][1]['attention_mask'])

Training

In [ ]:
from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model_pegasus
)

In [ ]:
from transformers import TrainingArguments, Trainer

trainer_args = TrainingArguments(
    output_dir='pegasus-samsum',
    num_train_epochs=3,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=500,
    save_steps=1e6,
    gradient_accumulation_steps=16
)

In [ ]:
trainer = Trainer(
    model=model_pegasus,
    args=trainer_args,
    processing_class=tokenizer,
    data_collator=seq2seq_data_collator,
    train_dataset=dataset_samsum_pt["train"],
    eval_dataset=dataset_samsum_pt["validation"],
)

In [ ]:
trainer.train()

Evaluation

In [ ]:
def generate_batch_size_chunks(list_of_elements, batch_size):
  for i in range(0, len(list_of_elements), batch_size):
    yield list_of_elements[i: i+batch_size]


def calculate_metric_on_test_ds(dataset, metric, model, tokenizer, batch_size=16, device=device, column_text="article", column_summary="highlighst"):
  article_batches = list(generate_batch_size_chunks(dataset[column_text], batch_size))
  target_batches = list(generate_batch_size_chunks(dataset[column_summary], batch_size))

  for article_batch, target_batch in tqdm(
      zip(article_batches, target_batches), total=len(article_batches)):
      inputs = tokenizer(article_batch, max_length=1024, truncation=True, padding="max_length", return_tensors="pt")
      summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                                 attention_mask=inputs["attention_mask"].to(device),
                                 length_penalty=0.8, num_beams=8, max_length=128
                                 )
      decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                            clean_up_tokenization_spaces=True) for s in summaries]
      decoded_summaries = [d.replace("\n", " ") for d in decoded_summaries]
      metric.add_batch(predictions=decoded_summaries, references=target_batch)
  score = metric.compute()
  return score


In [ ]:
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_metric = evaluate.load("rouge")

In [ ]:
score = calculate_metric_on_test_ds(
    ds['test'][0:10], rouge_metric, trainer.model, tokenizer, batch_size=2, column_text='dialogue', column_summary='summary')
rouge_dict = {rn: score[rn] for rn in rouge_names}
pd.DataFrame(rouge_dict, index=[f'pegasus'])

Save Model and Tokenizer

In [ ]:
model_pegasus.save_pretrained("pegasus-samsum-model")
tokenizer.save_pretrained("tokenizer")

Loading

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("/content/tokenizer")
model = AutoModelForSeq2SeqLM.from_pretrained( "/content/pegasus-samsum-model").to(device)


Prediction

In [ ]:
gen_kwargs = {
    "length_penalty": 0.8,
    "num_beams": 8,
    "max_length": 128
}

# Sample dialogue and reference summary
sample_text = ds["test"][0]["dialogue"]
reference = ds["test"][0]["summary"]

# Tokenize the dialogue
inputs = tokenizer(
    sample_text,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
)

# Move tensors to GPU (or CPU)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Put model in evaluation mode
trainer.model.eval()

# Disable gradient calculation during inference
with torch.no_grad():

    # Generate summary
    summary_ids = trainer.model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        **gen_kwargs
    )

# Decode summary
generated_summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

# Replace PEGASUS newline token
generated_summary = generated_summary.replace("<n>", "\n")

# Print everything
print(f"Dialogue:\n{sample_text}\n")
print(f"Reference:\n{reference}\n")
print("Generated Summary:\n")
print(generated_summary)